# Кластеризация по плотности: DBSCAN и HDBSCAN

> Модуль **Unsupervised Learning**, ноутбук 4. Справочный, академичный. Здесь — методы,
> которые **сами определяют число кластеров**, ловят кластеры **произвольной формы** и, что
> особенно ценно, умеют честно сказать «этот объект — **шум**».

### План занятия
1. **Зачем нужны плотностные методы** — что не умели K-Means и GMM.
2. **DBSCAN** — ядровые/граничные/шумовые точки, параметры `eps` и `min_samples`.
3. **Подбор `eps`** — график k-расстояний («колено»).
4. **Ограничения DBSCAN** — единый `eps` и разная плотность.
5. **HDBSCAN** — иерархическая версия без `eps`, устойчивая к разной плотности.
6. **Сравнение** K-Means / DBSCAN / HDBSCAN на сложных данных.
7. **Поиск шума и аномалий** — почему это ценно на практике (детекция выбросов, очистка данных,
   фрод, поломки сенсоров).
8. **Практика** — дикторы: HDBSCAN сам находит 3 группы и отмечает нетипичные записи.

В конце — раздел **вопросов и ответов**.

## 1. Зачем нужны плотностные методы

У K-Means и GMM есть три общих неудобства:

- **Нужно заранее задать число кластеров** ($k$ или число компонент).
- **Каждый объект обязан попасть в кластер.** Выброс «прилипает» к ближайшему центру и ещё и
  тянет его на себя — отдельного понятия «шум» нет.
- **Предпочитают выпуклые формы** (шары/эллипсы). Изогнутые кластеры (полумесяцы, спирали) им
  не даются.

**Плотностные методы** смотрят на задачу иначе: **кластер — это область высокой плотности
точек, окружённая разрежением**. Такой взгляд сразу даёт три вещи: число кластеров определяется
автоматически, форма может быть любой, а точки в разреженных областях честно помечаются как
**шум** (метка `-1`).

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from sklearn.cluster import DBSCAN, HDBSCAN, KMeans
from sklearn.datasets import make_moons, make_blobs
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

try:
    from ipywidgets import interact, FloatSlider, IntSlider
    HAVE_WIDGETS = True
except Exception:
    HAVE_WIDGETS = False

np.random.seed(0)
plt.rcParams.update({"figure.figsize": (7, 5), "axes.grid": True, "grid.alpha": 0.3})
PALETTE = ["#2f6f4f", "#2d5d8a", "#9b3b46", "#9a6b00", "#5b4b8a", "#3a7d7b", "#a0522d"]


def plot_clusters(ax, X, labels, title=""):
    # noise (label -1) is drawn as light-gray crosses
    for lab in sorted(set(labels)):
        m = labels == lab
        if lab == -1:
            ax.scatter(X[m, 0], X[m, 1], c="lightgray", s=14, marker="x", label="шум")
        else:
            ax.scatter(X[m, 0], X[m, 1], c=PALETTE[lab % len(PALETTE)], s=14, alpha=0.7)
    ax.set_title(title)


# Datasets we will reuse
X_moons, y_moons = make_moons(n_samples=600, noise=0.06, random_state=0)
rng = np.random.default_rng(0)
X_var = np.vstack([rng.normal([0, 0], 0.30, (250, 2)),     # tight
                   rng.normal([3, 0], 0.30, (250, 2)),     # tight
                   rng.normal([1.5, 3], 1.10, (250, 2))])  # loose
y_var = np.r_[np.zeros(250), np.ones(250), np.full(250, 2)].astype(int)
print("datasets ready | виджеты:", "да" if HAVE_WIDGETS else "нет")

## 2. DBSCAN (Density-Based Spatial Clustering of Applications with Noise)

DBSCAN формализует «плотность» через два параметра:

- **`eps`** ($\varepsilon$) — радиус окрестности точки;
- **`min_samples`** — сколько соседей (включая саму точку) должно попасть в радиус `eps`,
  чтобы окрестность считалась «плотной».

По этим параметрам каждая точка относится к одному из трёх типов:

| Тип точки | Условие | Роль |
|---|---|---|
| **Ядровая** (core) | в её $\varepsilon$-окрестности $\ge$ `min_samples` точек | «сердце» кластера |
| **Граничная** (border) | сама не ядровая, но попадает в окрестность ядровой | край кластера |
| **Шум** (noise) | ни то, ни другое | выброс, метка `-1` |

**Как растёт кластер.** Берём ядровую точку, забираем в кластер всех её соседей; если среди них
есть другие ядровые — продолжаем расширяться через них (это называют *плотностной
достижимостью*). Так кластер «перетекает» по плотной области любой формы, пока не упрётся в
разрежение. Точки, до которых «дотянуться» не удалось, остаются шумом.

> **Ключевое отличие от K-Means:** число кластеров **не задаётся** — сколько найдётся плотных
> областей, столько и будет; а выбросы не портят центры, потому что просто помечаются шумом.

### Как выглядят и как детектятся ядровые, граничные и шумовые точки

Разберём механику на маленьком наборе. Зафиксируем `eps` и `min_samples` и вокруг каждой точки
мысленно очертим круг радиуса `eps`. Дальше — простое правило подсчёта соседей в этом круге:

- **ядровая** (зелёная заливка): в круге **не меньше `min_samples`** точек — точка внутри
  плотной области;
- **граничная** (синий контур): соседей меньше `min_samples`, но сама точка попала в круг
  какой-то ядровой — это «край» кластера;
- **шум** (серый крестик, метка `−1`): и не ядро, и не дотянулась до ядра — одинокая точка.

Ниже — **интерактивная** схема. Четыре опорные точки **A–D** стоят в местах разной плотности;
вокруг каждой строится `eps`-окрестность, а в панели слева видно, **сколько соседей** попало в
радиус и какой из-за этого **статус**. Крутите ползунки `eps` и `min_samples` и наблюдайте, как
точки перетекают между статусами «ядро → граница → шум»:

In [ ]:
# Interactive anatomy: move eps / min_samples and watch the statuses of points A-D change
from matplotlib.patches import Circle
from scipy.spatial.distance import cdist

rng_a = np.random.default_rng(3)
X_demo, _ = make_blobs(n_samples=90, centers=2, cluster_std=0.55, random_state=3)
X_demo = np.vstack([X_demo, rng_a.uniform(X_demo.min(0) - 1.2, X_demo.max(0) + 1.2, (6, 2))])
N = len(X_demo)
D = cdist(X_demo, X_demo)                       # pairwise distances (dataset is small)

# Four reference points of different local density: dense / medium / edge / isolated
density5 = np.sort(D, axis=1)[:, 5]             # distance to 5th neighbour (smaller = denser)
order = np.argsort(density5)
REFS = [int(order[2]), int(order[N // 2]), int(order[int(N * 0.82)]), N - 1]
REF_NAMES = ["A", "B", "C", "D"]
STATUS_COLOR = {"ядро": "#2f6f4f", "граница": "#2d5d8a", "шум": "#9b3b46"}


def anatomy(eps=0.5, min_samples=5):
    db = DBSCAN(eps=eps, min_samples=min_samples).fit(X_demo)
    is_core = np.zeros(N, bool); is_core[db.core_sample_indices_] = True
    is_noise = db.labels_ == -1
    is_border = ~is_core & ~is_noise
    counts = (D <= eps).sum(1)                  # neighbours within eps (incl. self)
    status = lambda i: "ядро" if is_core[i] else ("граница" if is_border[i] else "шум")

    fig, ax = plt.subplots(figsize=(11, 8.5))
    ax.scatter(X_demo[is_core, 0],   X_demo[is_core, 1],   s=70, c="#2f6f4f", alpha=0.85,
               label=f"ядровые: ≥ {min_samples} соседей")
    ax.scatter(X_demo[is_border, 0], X_demo[is_border, 1], s=70, facecolor="white",
               edgecolor="#2d5d8a", linewidth=2, label="граничные")
    ax.scatter(X_demo[is_noise, 0],  X_demo[is_noise, 1],  s=80, c="lightgray", marker="x",
               label="шум (метка −1)")

    # reference points: eps-circle + big labelled marker (statuses go in a side panel)
    for name, i in zip(REF_NAMES, REFS):
        col = STATUS_COLOR[status(i)]
        ax.add_patch(Circle(X_demo[i], eps, fill=False, edgecolor=col, lw=2, ls="--", alpha=0.9))
        ax.scatter(*X_demo[i], s=330, facecolor="white", edgecolor="black", linewidth=1.6, zorder=5)
        ax.text(*X_demo[i], name, ha="center", va="center", fontsize=12, fontweight="bold", zorder=6)

    # readable info panel: one coloured line per reference point (top-left, off the points)
    for row, (name, i) in enumerate(zip(REF_NAMES, REFS)):
        st = status(i); col = STATUS_COLOR[st]
        ax.text(0.015, 0.975 - row * 0.055,
                f"{name}:  {counts[i]} соседей в радиусе  →  {st}",
                transform=ax.transAxes, fontsize=11, fontweight="bold", color=col, va="top",
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=col, alpha=0.95))

    ax.set_aspect("equal"); ax.legend(loc="lower right", fontsize=10, framealpha=0.9)
    ax.set_title(f"Анатомия DBSCAN  ·  eps={eps:.2f}, min_samples={min_samples}  ·  "
                 f"ядро={is_core.sum()}, граница={is_border.sum()}, шум={is_noise.sum()}", fontsize=12)
    plt.show()


if HAVE_WIDGETS:
    interact(anatomy,
             eps=FloatSlider(min=0.2, max=1.3, step=0.05, value=0.5, description="eps"),
             min_samples=IntSlider(min=2, max=10, step=1, value=5, description="min_samples"))
else:
    anatomy(0.5, 5)

Теперь — интерактивно. Покрутите `eps` на «полумесяцах» (форма, на которой K-Means ломался), и
проследите, как меняются и число кластеров, и объём шума:

In [ ]:
def dbscan_eps(eps=0.20):
    labels = DBSCAN(eps=eps, min_samples=5).fit_predict(X_moons)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = int((labels == -1).sum())
    fig, ax = plt.subplots(figsize=(7, 6))
    plot_clusters(ax, X_moons, labels)
    ax.set_title(f"DBSCAN, eps={eps:.2f}  ·  кластеров: {n_clusters}  ·  шум: {n_noise}")
    if n_noise:
        ax.legend(loc="upper right")
    plt.show()


if HAVE_WIDGETS:
    interact(dbscan_eps, eps=FloatSlider(min=0.05, max=0.6, step=0.01, value=0.20))
else:
    dbscan_eps(0.20)

Обратите внимание на поведение: **слишком маленький `eps`** — почти всё становится шумом;
**слишком большой** — два полумесяца сливаются в один кластер; **в правильном диапазоне** — два
аккуратных изогнутых кластера, которые K-Means никогда бы не нашёл.

### Разные формы — и везде есть шум

Главная сила плотностного подхода — кластеры **любой формы**, а выбросы честно отсекаются.
Подмешаем равномерный шум в несколько разных наборов и посмотрим, как DBSCAN (с подобранным под
каждый набор `eps`) находит форму и помечает шум серыми крестиками:

In [ ]:
from sklearn.datasets import make_circles


def add_noise(X, k, seed):
    r = np.random.default_rng(seed)
    return np.vstack([X, r.uniform(X.min(0) - 0.5, X.max(0) + 0.5, (k, 2))])


shapes = [
    ("Концентрические кольца",
     add_noise(make_circles(400, factor=0.45, noise=0.04, random_state=0)[0] * 4, 40, 1), 0.6),
    ("Полумесяцы",
     add_noise(make_moons(400, noise=0.06, random_state=0)[0] * 3, 40, 2), 0.4),
    ("Вытянутые (anisotropic)",
     add_noise(make_blobs(600, random_state=170)[0] @ np.array([[0.6, -0.6], [-0.4, 0.8]]), 50, 3), 0.4),
    ("Три блоба",
     add_noise(make_blobs(500, centers=3, cluster_std=0.6, random_state=2)[0], 50, 4), 0.6),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.6))
for ax, (name, X, eps) in zip(axes, shapes):
    labels = DBSCAN(eps=eps, min_samples=5).fit_predict(X)
    nc = len(set(labels)) - (1 if -1 in labels else 0)
    plot_clusters(ax, X, labels, f"{name}\neps={eps}: {nc} кл., шум {int((labels==-1).sum())}")
plt.tight_layout()
plt.show()

## 3. Как подобрать `eps`: график k-расстояний

Удобная эвристика. Зафиксируем $k = $ `min_samples` и для каждой точки посчитаем расстояние до
её **$k$-го ближайшего соседа**. Отсортируем эти расстояния по возрастанию и построим график.
У «правильного» `eps` на графике виден **изгиб («колено»)**: слева — плотные точки внутри
кластеров (близкие соседи), справа — резкий взлёт для шумовых точек. Значение в колене и берут
за `eps`.

Про `min_samples`: эвристика — брать не меньше `d + 1` (а часто `2*d`), где `d` — размерность;
для 2D обычно 4–5. Больше `min_samples` → строже определение плотности → больше шума.

In [ ]:
k = 5
nn = NearestNeighbors(n_neighbors=k).fit(X_moons)
dist, _ = nn.kneighbors(X_moons)
kdist = np.sort(dist[:, -1])               # distance to the k-th neighbour, sorted

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(kdist, color="#2d5d8a")
ax.axhline(0.20, color="#9b3b46", ls="--", label="колено ≈ 0.20 → eps")
ax.set_xlabel("точки, отсортированы по k-расстоянию")
ax.set_ylabel(f"расстояние до {k}-го соседа")
ax.set_title("График k-расстояний: ищем «колено»")
ax.legend()
plt.show()

## 4. Ограничение DBSCAN: единый `eps` и разная плотность

У DBSCAN **один** `eps` на весь набор. Если в данных есть кластеры **разной плотности** (одни
плотные, другие рыхлые), единого радиуса, который подошёл бы всем, может просто **не
существовать**: маленький `eps` разорвёт рыхлый кластер на шум, большой — склеит плотные в один.
Покажем это на данных с двумя плотными и одним рыхлым кластером:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, eps in zip(axes, [0.3, 0.7]):
    labels = DBSCAN(eps=eps, min_samples=5).fit_predict(X_var)
    nc = len(set(labels)) - (1 if -1 in labels else 0)
    plot_clusters(ax, X_var, labels, f"eps={eps}: кластеров {nc}, шум {int((labels==-1).sum())}")
plt.show()
print("Ни маленький, ни большой eps не дают трёх честных кластеров — нужен HDBSCAN.")

## 5. HDBSCAN (Hierarchical DBSCAN)

HDBSCAN снимает проблему единого `eps`: он, по сути, прогоняет DBSCAN **сразу для всех значений
`eps`** и строит **иерархию** вложенных кластеров (от «всё — один кластер» при большом радиусе
до «всё рассыпалось» при маленьком). Затем из этой иерархии выбираются **самые устойчивые**
кластеры — те, что «живут» в широком диапазоне плотностей. Поэтому разные кластеры могут быть
выделены при *разной* эффективной плотности — то, чего DBSCAN не умеет.

Главный параметр — **`min_cluster_size`**: минимальный размер группы, которую мы согласны
считать кластером (а не случайным сгустком). `eps` задавать **не нужно**. Бонусом HDBSCAN
выдаёт `probabilities_` — насколько уверенно точка принадлежит своему кластеру (у шума — 0).

Сравним прямо на тех же разноплотных данных, где DBSCAN спотыкался:

In [ ]:
def hdbscan_mcs(min_cluster_size=20):
    h = HDBSCAN(min_cluster_size=min_cluster_size).fit(X_var)
    labels = h.labels_
    nc = len(set(labels)) - (1 if -1 in labels else 0)
    fig, ax = plt.subplots(figsize=(7, 6))
    plot_clusters(ax, X_var, labels)
    ax.set_title(f"HDBSCAN, min_cluster_size={min_cluster_size}  ·  "
                 f"кластеров: {nc}  ·  шум: {int((labels==-1).sum())}")
    ax.legend(loc="upper right")
    plt.show()


if HAVE_WIDGETS:
    interact(hdbscan_mcs, min_cluster_size=IntSlider(min=5, max=80, step=5, value=20))
else:
    hdbscan_mcs(20)

## 6. Сравнение: K-Means vs DBSCAN vs HDBSCAN

Соберём всё вместе на трёх «неудобных» наборах: полумесяцы (форма), разная плотность и
компактные блобы с подмешанным шумом. Серые крестики — то, что метод пометил как шум.

In [ ]:
# blobs + injected uniform noise
Xb, _ = make_blobs(n_samples=500, centers=3, cluster_std=0.55, random_state=0)
lo, hi = Xb.min(0) - 2, Xb.max(0) + 2
Xb = np.vstack([Xb, rng.uniform(lo, hi, size=(70, 2))])

datasets = [("Полумесяцы", X_moons, 2, 0.20, 5),
            ("Разная плотность", X_var, 3, 0.3, 5),
            ("Блобы + шум", Xb, 3, 0.6, 5)]
methods = ["K-Means", "DBSCAN", "HDBSCAN"]

fig, axes = plt.subplots(3, 3, figsize=(14, 13))
for col, (name, X, k, eps, ms) in enumerate(datasets):
    km = KMeans(k, n_init=10, random_state=0).fit_predict(X)
    db = DBSCAN(eps=eps, min_samples=ms).fit_predict(X)
    hd = HDBSCAN(min_cluster_size=20).fit_predict(X)
    for row, (m, lab) in enumerate(zip(methods, [km, db, hd])):
        plot_clusters(axes[row, col], X, lab, f"{m} — {name}")
plt.tight_layout()
plt.show()

## 7. Поиск шума и аномалий — и чем это ценно на практике

Способность плотностных методов помечать точки как **шум** (`-1`) — это не побочный эффект, а
самостоятельный, очень практичный инструмент. K-Means и GMM так не умеют: они *обязаны* отнести
каждый объект к какому-то кластеру, поэтому выброс там «прячется» внутри группы. DBSCAN/HDBSCAN
же прямо говорят: «эта точка не вписывается ни в одну плотную область».

**Где это ценно в реальных задачах:**

- **Очистка данных перед обучением.** Битые строки, опечатки, артефакты сбора — выбросы в
  пространстве признаков. Удалив их, мы не даём модели учиться на мусоре.
- **Детекция мошенничества (fraud).** Аномальная транзакция не похожа на «плотную массу»
  обычных — она оказывается в разрежении и помечается шумом.
- **Поломки и сбои оборудования.** Показания неисправного датчика выбиваются из нормального
  режима работы.
- **Сетевые вторжения / безопасность.** Необычный трафик — выброс на фоне типичного.
- **Контроль качества и поиск дефектов**, медицинская диагностика редких случаев, поиск
  нетипичных пользователей/товаров.

Идея одна: **«нормальное» обычно плотное и многочисленное, аномальное — редкое и одинокое.**
Покажем на блобах с подмешанным шумом, насколько точно DBSCAN ловит выбросы:

In [ ]:
# Blobs + known injected noise: measure how well DBSCAN flags the noise
rng2 = np.random.default_rng(7)
Xc, _ = make_blobs(n_samples=500, centers=3, cluster_std=0.55, random_state=0)
lo, hi = Xc.min(0) - 2, Xc.max(0) + 2
noise_pts = rng2.uniform(lo, hi, size=(60, 2))
X_an = np.vstack([Xc, noise_pts])
is_noise = np.r_[np.zeros(500), np.ones(60)].astype(bool)   # ground truth

db = DBSCAN(eps=0.6, min_samples=5).fit(X_an)
flagged = db.labels_ == -1
tp = int((flagged & is_noise).sum())
precision = tp / max(flagged.sum(), 1)
recall = tp / is_noise.sum()
print(f"DBSCAN как детектор выбросов:  precision={precision:.2f}  recall={recall:.2f}  "
      f"(поймано {tp} из {is_noise.sum()} истинных выбросов)")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
a1.scatter(X_an[~is_noise, 0], X_an[~is_noise, 1], s=12, color="#2d5d8a", alpha=0.5, label="норма")
a1.scatter(X_an[is_noise, 0], X_an[is_noise, 1], s=25, color="#9b3b46", label="истинный шум")
a1.legend(); a1.set_title("Истина: где подмешан шум")
a2.scatter(X_an[~flagged, 0], X_an[~flagged, 1], s=12, color="#2d5d8a", alpha=0.5, label="кластеры")
a2.scatter(X_an[flagged, 0], X_an[flagged, 1], s=25, color="#9b3b46", marker="x", label="помечено шумом")
a2.legend(); a2.set_title("DBSCAN: что помечено выбросом")
plt.show()

**Мягкий «скор аномальности».** HDBSCAN, помимо жёсткой метки `-1`, даёт `probabilities_` —
силу принадлежности кластеру. Величину $1 - \text{probability}$ удобно использовать как
**непрерывный балл аномальности**: чем выше, тем «более чужая» точка. Это позволяет не просто
делить на «норму/шум», а **ранжировать** объекты по подозрительности и разбирать самые
сомнительные первыми.

> На заметку: в отдельной библиотеке `hdbscan` есть готовый `outlier_scores_` (алгоритм GLOSH)
> — специализированный балл аномальности. В `scikit-learn` его пока нет, но $1-$`probabilities_`
> служит хорошим практичным приближением.

In [ ]:
h = HDBSCAN(min_cluster_size=20).fit(X_an)
anomaly_score = 1.0 - h.probabilities_           # higher = more anomalous

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(X_an[:, 0], X_an[:, 1], c=anomaly_score, cmap="inferno", s=20)
plt.colorbar(sc, label="балл аномальности  (1 − probability)")
ax.set_title("HDBSCAN: непрерывный балл аномальности")
plt.show()
print("Топ по аномальности — это и есть кандидаты в выбросы для ручного разбора.")

## 8. Практика: дикторы и нетипичные записи

Запустим HDBSCAN на эмбеддингах дикторов. Мы **не задаём число кластеров** — алгоритм сам решит,
сколько их, и заодно пометит записи, которые не вписываются ни в одну группу (это могут быть
зашумлённые или нетипичные аудио).

In [ ]:
def load_speaker_embeddings(candidates=("speaker_emb.npz", "data/speaker_emb.npz")):
    # Local cache first; otherwise pull ONLY embedding/label columns from HuggingFace
    # (skips the 2.65 GB audio), then cache. One-time remote step ~1-2 min, then instant.
    import os
    for c in candidates:
        if os.path.exists(c):
            d = np.load(c, allow_pickle=True)
            return d["X"], d["y"], d["text"]
    from huggingface_hub import HfApi
    import pyarrow.parquet as pq
    import fsspec
    repo = "aiacademy-kg/audio-3-speaker-dataset"
    files = [f for f in HfApi().list_repo_files(repo, repo_type="dataset") if f.endswith(".parquet")]
    fs = fsspec.filesystem("hf")
    E, Y, T = [], [], []
    for f in files:
        pf = pq.ParquetFile(fs.open(f"datasets/{repo}/{f}"))
        d = pf.read(columns=["wavlm_embedding", "speaker_id", "text"]).to_pandas()
        E.append(np.array(d["wavlm_embedding"].tolist(), dtype=np.float32))
        Y += d["speaker_id"].tolist(); T += d["text"].tolist()
    X, y, text = np.vstack(E), np.array(Y), np.array(T, dtype=object)
    np.savez_compressed("speaker_emb.npz", X=X, y=y, text=text)
    return X, y, text


X_emb, y_emb, _ = load_speaker_embeddings()
Pe = PCA(n_components=10, random_state=0).fit_transform(StandardScaler().fit_transform(X_emb))

hd = HDBSCAN(min_cluster_size=50).fit(Pe)
lab = hd.labels_
n_clusters = len(set(lab)) - (1 if -1 in lab else 0)
mask = lab != -1
print("Найдено кластеров (без подсказки!):", n_clusters, "| истинных дикторов: 3")
print("Помечено шумом (нетипичные записи):", int((lab == -1).sum()), "из", len(lab))
print("ARI на не-шумовых записях:", round(adjusted_rand_score(y_emb[mask], lab[mask]), 3))
print("\nДиктор × кластер (−1 = шум):")
print(pd.crosstab(y_emb, lab))

## 9. Выводы

- **Плотностные методы** видят кластер как плотную область среди разрежения: число кластеров
  не задаётся, форма любая, выбросы помечаются **шумом** (`-1`).
- **DBSCAN**: параметры `eps` и `min_samples`; точки делятся на ядровые/граничные/шумовые;
  `eps` подбирают по «колену» графика k-расстояний.
- **Ограничение DBSCAN** — единый `eps`: на кластерах разной плотности не работает.
- **HDBSCAN** строит иерархию по всем `eps` и берёт самые устойчивые кластеры; параметр
  `min_cluster_size`, `eps` не нужен; справляется с разной плотностью и даёт `probabilities_`.
- **Поиск шума/аномалий** — самостоятельная ценность: очистка данных, фрод, поломки, безопасность.
  $1-$`probabilities_` — практичный непрерывный балл аномальности.
- На эмбеддингах **HDBSCAN сам нашёл 3 кластера** и пометил несколько нетипичных записей; ARI на
  остальных ≈ 1.

**Дальше:** иерархическая кластеризация и **дендрограммы** — ещё один взгляд, где кластеры
образуют дерево вложенности, а число групп выбирают, «разрезая» это дерево.

## Вопросы и ответы

**1) Чем DBSCAN принципиально отличается от K-Means?**
Не нужно задавать число кластеров; ловит любые формы; помечает выбросы шумом, а не запихивает их
в кластеры. Зато чувствителен к `eps` и плохо работает при разной плотности.

**2) Как понять, что `eps` выбран плохо?**
Почти всё ушло в шум — `eps` мал; всё слилось в один кластер — `eps` велик. Стройте график
k-расстояний и берите значение в «колене».

**3) Когда брать HDBSCAN вместо DBSCAN?**
Когда кластеры разной плотности, когда не хочется подбирать `eps`, и когда нужны вероятности
принадлежности. HDBSCAN, как правило, удобнее «из коробки».

**4) Плотностные методы работают в высокой размерности?**
С трудом: из-за «проклятия размерности» расстояния теряют контраст, и понятие плотности
размывается. Практичный приём — **сначала снизить размерность** (PCA/UMAP), как мы делаем с
эмбеддингами, а уже потом кластеризовать.

**5) Шум — это всегда «плохие» данные?**
Нет. Иногда шум — это **самое интересное** (мошенничество, поломка, редкое событие). «Норма»
плотная, аномалии редки и одиноки — поэтому плотностная кластеризация и используется как
детектор аномалий.

**6) Почему HDBSCAN иногда метит шумом точки на краю явного кластера?**
Край кластера менее плотный, и при выбранном `min_cluster_size` пограничные точки могут не
дотянуть до устойчивого кластера. Это регулируется `min_cluster_size` и `min_samples`.